# NLP Assignment 2025/26 - PoliMillionaire QuizBot Steroids

**Task.** Build and evaluate local open-weight chatbot models for the online quiz *Who wants to be a PoliMillionaire?*.

**Group members.** TODO: add names and Polimi email addresses.

**Video link.** TODO: add the screen-capture video link.

**Coding assistants statement.** TODO: state which coding assistants, if any, were used.

This notebook uses the official `millionaire_client` package provided with the course. This steroids version keeps text mode and no RAG, but uses category-specific prompts, two local quantized models, and a local judge tie-breaker when the models disagree.

## 1. Global Settings

All operational settings are centralized here. The rest of the notebook should run top-to-bottom without changing local flags inside later cells.

In [ ]:
# Centralized, all settings are. Scattered through the notebook, they are not.
API_URL = 'http://131.175.15.22:51111/'
USERNAME = 'MarianoAkaMery'
PASSWORD = 'Test1234!'

DRIVE_PROJECT_DIR = '/content/gdrive/MyDrive/[2025-26] - 088946 - NLP PROJECT'
COMPETITION_KEY = 'ancient_history_politics'
COMPETITIONS = {
    'entertainment': {
        'id': 0,
        'name': 'Entertainment',
        'description': 'Music, Movies, Celebrities and more',
    },
    'ancient_history_politics': {
        'id': 1,
        'name': 'Ancient History and Politics',
        'description': 'The Roman Empire, The Greeks, and more',
    },
    'science_nature': {
        'id': 2,
        'name': 'Science and Nature',
        'description': 'Chemistry, Biology, Physics and similar subjects',
    },
    'maths': {
        'id': 3,
        'name': 'Maths',
        'description': 'Mathematics and Statistics from High School and College',
    },
}
COMPETITION_ID = COMPETITIONS[COMPETITION_KEY]['id']
GAME_MODE = 'text'

MODEL_A_NAME = 'Qwen/Qwen2.5-3B-Instruct'
MODEL_B_NAME = 'Qwen/Qwen2.5-1.5B-Instruct'
LOAD_IN_4BIT = True
USE_ENSEMBLE = True
PREFER_MODEL_B_ON_DISAGREEMENT = True
USE_JUDGE_ON_DISAGREEMENT = True
USE_CALCULATOR_FOR_MATHS = True
MAX_NEW_TOKENS = 4

MAX_LEVELS_TO_PLAY = 15
DELAY_BETWEEN_QUESTIONS_S = 1.0
SAVE_RUN_LOG = True

INSTALL_MODEL_DEPENDENCIES = True
RUN_API_CHECK = True
RUN_MODEL_WARMUP = True
RUN_FULL_GAME = True

print('Settings loaded.')

## 2. Colab Setup

This mounts Google Drive, locates the project folder, and makes the official client package importable.

In [ ]:
# Mounted, Google Drive is. Found, the project folder must be.
import os
import sys
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/gdrive/')
    IN_COLAB = True
except Exception:
    IN_COLAB = False

PROJECT_DIR = DRIVE_PROJECT_DIR if IN_COLAB else os.getcwd()
if PROJECT_DIR not in sys.path:
    sys.path.append(PROJECT_DIR)

print('IN_COLAB:', IN_COLAB)
print('PROJECT_DIR:', PROJECT_DIR)
print('millionaire_client exists:', os.path.isdir(os.path.join(PROJECT_DIR, 'millionaire_client')))

## 3. Dependencies

The API client only needs `requests`, already available in most environments. The local models need `transformers`, `accelerate`, `sentencepiece`, and `torch`.

In [ ]:
# Installed, model dependencies are. Local models, then usable become.
if INSTALL_MODEL_DEPENDENCIES:
    import subprocess
    subprocess.check_call([
        sys.executable,
        '-m',
        'pip',
        'install',
        '-q',
        'transformers',
        'accelerate',
        'sentencepiece',
        'bitsandbytes',
        'sympy',
        'torch'
    ])
    print('Model dependencies installed or already available.')
else:
    print('Dependency installation skipped by settings.')

import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING: no GPU detected. CPU inference will likely timeout.')

## 4. Official Game Client

This section logs in and checks the game API using only the course-provided `millionaire_client` package.

In [ ]:
# Imported, the official client is. Invented endpoints, used they are not.
from millionaire_client import MillionaireClient, AuthenticationError, MillionaireError

client = MillionaireClient(API_URL)
user = client.login(USERNAME, PASSWORD)
print(f'Logged in as {user.username} (role={user.role})')

if RUN_API_CHECK:
    competitions = client.competitions.list_all()
    print('\nAvailable competitions:')
    for comp in competitions:
        print(f'  {comp.id}: {comp.name} | max_levels={comp.max_levels} | questions={comp.question_count}')
else:
    print('API check skipped by settings.')

## 5. Category Prompts and Output Parsing

The model receives category-specific instructions and answers with a letter. The notebook then maps that letter back to the official `option_id` required by the API.

In [ ]:
# Built, category-aware prompts are. Parsed, only valid answer letters are.
import re

CATEGORY_PROMPTS = {
    'entertainment': {
        'system': (
            'You are a careful quiz expert on music, movies, television, celebrities, awards, '
            'popular culture, and entertainment history. Prefer widely accepted factual knowledge. '
            'Watch for dates, names, titles, casts, directors, singers, and awards.'
        ),
        'examples': [
            ('Which band released Bohemian Rhapsody?', ['The Beatles', 'Queen', 'Led Zeppelin', 'ABBA'], 'B'),
        ],
    },
    'ancient_history_politics': {
        'system': (
            'You are a careful quiz expert on ancient history and politics, especially Rome, Greece, '
            'empires, republics, institutions, rulers, wars, chronology, and political offices. '
            'Pay attention to wording such as NOT, best describes, before, after, and according to the passage.'
        ),
        'examples': [
            ('In what year is the fall of the Western Roman Empire traditionally dated?', ['410 AD', '455 AD', '476 AD', '500 AD'], 'C'),
        ],
    },
    'science_nature': {
        'system': (
            'You are a careful quiz expert on science and nature, including chemistry, biology, physics, '
            'earth science, astronomy, units, constants, and scientific terminology. Use standard textbook facts.'
        ),
        'examples': [
            ('What is the chemical symbol for gold?', ['Go', 'Gd', 'Au', 'Ag'], 'C'),
        ],
    },
    'maths': {
        'system': (
            'You are a careful mathematics and statistics tutor. Compute arithmetic exactly, compare units, '
            'and choose the option matching the result. Return only the final letter.'
        ),
        'examples': [
            ('What is 15% of 200?', ['25', '30', '35', '40'], 'B'),
        ],
    },
}

def option_lines(options):
    return '\n'.join(f'{chr(65 + idx)}) {opt.text}' for idx, opt in enumerate(options))

def example_block(examples):
    blocks = []
    for question_text, options, answer in examples:
        opts = '\n'.join(f'{chr(65 + idx)}) {text}' for idx, text in enumerate(options))
        blocks.append(f'Question: {question_text}\n{opts}\nAnswer: {answer}')
    return '\n\n'.join(blocks)

def build_prompt(question):
    config = CATEGORY_PROMPTS[COMPETITION_KEY]
    examples = example_block(config['examples'])
    return f"""{config['system']}

You are playing a multiple-choice quiz.
Choose the single best option.
Return only one letter: A, B, C, or D.
Do not explain.

Examples:
{examples}

Question: {question.text}
{option_lines(question.options)}
Answer:"""

def build_judge_prompt(question, pred_a, pred_b):
    config = CATEGORY_PROMPTS[COMPETITION_KEY]
    return f"""{config['system']}

Two local models answered the same multiple-choice quiz question and disagreed.
Your job is to choose the most likely correct answer.
Return only one letter: A, B, C, or D.
Do not explain.

Question: {question.text}
{option_lines(question.options)}

Model A answered: {pred_a['letter']} ({pred_a['raw']})
Model B answered: {pred_b['letter']} ({pred_b['raw']})

Final answer:"""

def parse_answer_letter(raw_text, question):
    valid_letters = [chr(65 + idx) for idx in range(len(question.options))]
    cleaned = str(raw_text).strip().upper()
    if cleaned:
        first = re.sub(r'[^A-Z]', '', cleaned[:8])
        if first and first[0] in valid_letters:
            return first[0]
    for char in cleaned:
        if char in valid_letters:
            return char
    return None

def letter_to_option_id(letter, question):
    if letter is None:
        return None
    index = ord(letter) - ord('A')
    if 0 <= index < len(question.options):
        return question.options[index].id
    return None

def try_calculator(question):
    if COMPETITION_KEY != 'maths' or not USE_CALCULATOR_FOR_MATHS:
        return None

    from sympy import N as sympy_n
    from sympy import sympify

    text = question.text
    candidates = re.findall(r'[\d\.]+(?:\s*[\+\-\*\/\^]\s*[\d\.]+)+', text)
    percent_match = re.search(r'(\d+(?:\.\d+)?)\s*%\s*of\s*(\d+(?:\.\d+)?)', text, re.I)
    if percent_match:
        candidates.append(f'{percent_match.group(1)} * {percent_match.group(2)} / 100')

    for raw_expr in candidates:
        expr = raw_expr.strip().replace('^', '**').replace(' ', '')
        try:
            result = float(sympy_n(sympify(expr)))
        except Exception:
            continue

        for index, option in enumerate(question.options):
            option_number = re.sub(r'[^\d\.\-]', '', str(option.text))
            if not option_number:
                continue
            try:
                if abs(float(option_number) - result) < max(0.01, abs(result) * 0.001):
                    letter = chr(65 + index)
                    return {
                        'model': 'calculator',
                        'raw': f'{expr} = {result}',
                        'letter': letter,
                        'option_id': option.id,
                        'elapsed_s': 0.0,
                    }
            except Exception:
                continue

    return None

## 6. Local Model Wrapper

This wrapper loads an open-weight Hugging Face causal language model locally inside Colab. No LLM API is used.

In [ ]:
# Loaded locally, the model is. Called, no external LLM API is.
import time

class LocalCausalLMAnswerer:
    def __init__(self, model_name, max_new_tokens=8):
        self.model_name = model_name
        self.max_new_tokens = max_new_tokens
        self.tokenizer = None
        self.model = None

    def load(self):
        import torch
        from transformers import AutoModelForCausalLM, AutoTokenizer
        from transformers import BitsAndBytesConfig

        if self.model is not None:
            return

        self.tokenizer = AutoTokenizer.from_pretrained(self.model_name)
        quantization_config = None
        if LOAD_IN_4BIT:
            quantization_config = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_use_double_quant=True,
                bnb_4bit_quant_type='nf4',
                bnb_4bit_compute_dtype=torch.float16,
            )

        self.model = AutoModelForCausalLM.from_pretrained(
            self.model_name,
            device_map={'': 0} if torch.cuda.is_available() else 'cpu',
            torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
            quantization_config=quantization_config,
        )
        self.model.eval()

    def answer_from_prompt(self, prompt, question, max_new_tokens=None):
        self.load()
        inputs = self.tokenizer(prompt, return_tensors='pt').to(self.model.device)

        start = time.time()
        output_ids = self.model.generate(
            **inputs,
            max_new_tokens=max_new_tokens or self.max_new_tokens,
            do_sample=False,
            pad_token_id=self.tokenizer.eos_token_id
        )
        elapsed_s = time.time() - start

        generated_ids = output_ids[0][inputs['input_ids'].shape[-1]:]
        raw = self.tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

        letter = parse_answer_letter(raw, question)
        option_id = letter_to_option_id(letter, question)

        return {
            'model': self.model_name,
            'raw': raw,
            'letter': letter,
            'option_id': option_id,
            'elapsed_s': elapsed_s,
        }

    def answer(self, question):
        return self.answer_from_prompt(build_prompt(question), question)

model_a = LocalCausalLMAnswerer(MODEL_A_NAME, max_new_tokens=MAX_NEW_TOKENS)
model_b = LocalCausalLMAnswerer(MODEL_B_NAME, max_new_tokens=MAX_NEW_TOKENS)

print('Model A:', MODEL_A_NAME)
print('Model B:', MODEL_B_NAME)

## 7. Model Warm-up

The models must be loaded before starting a real game. If model weights are loaded during a question, the 30-second timer will likely expire.

In [ ]:
# Warmed up, models are. Timed out by loading weights, the game should not be.
import gc

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

if RUN_MODEL_WARMUP:
    print('Loading model A...')
    model_a.load()
    print('Model A ready.')

    if USE_ENSEMBLE:
        print('Loading model B...')
        model_b.load()
        print('Model B ready.')

    print('Warm-up complete.')
else:
    print('Model warm-up skipped by settings.')

## 8. Answer Strategy

The baseline can run either a single model or a two-model ensemble. In the ensemble setting, agreement is accepted; disagreement is resolved by preferring model B, the larger model.

In [ ]:
# Chosen, the final answer is. Stronger, the tie-breaker becomes.
def choose_answer(question):
    calc_pred = try_calculator(question)
    if calc_pred is not None:
        return {
            'chosen_option_id': calc_pred['option_id'],
            'chosen_letter': calc_pred['letter'],
            'decision': 'calculator',
            'pred_a': calc_pred,
            'pred_b': None,
            'pred_judge': None,
        }

    pred_a = model_a.answer(question)

    if not USE_ENSEMBLE:
        chosen = pred_a['option_id'] if pred_a['option_id'] is not None else question.options[0].id
        return {
            'chosen_option_id': chosen,
            'chosen_letter': pred_a['letter'] if pred_a['letter'] is not None else 'A',
            'decision': 'single_model_a',
            'pred_a': pred_a,
            'pred_b': None,
            'pred_judge': None,
        }

    pred_b = model_b.answer(question)
    pred_judge = None

    if pred_a['option_id'] is not None and pred_a['option_id'] == pred_b['option_id']:
        chosen = pred_a['option_id']
        decision = 'models_agree'
        chosen_letter = pred_a['letter']
    elif USE_JUDGE_ON_DISAGREEMENT:
        judge_prompt = build_judge_prompt(question, pred_a, pred_b)
        pred_judge = model_a.answer_from_prompt(judge_prompt, question, max_new_tokens=MAX_NEW_TOKENS)
        if pred_judge['option_id'] is not None:
            chosen = pred_judge['option_id']
            decision = 'judge_tiebreaker'
            chosen_letter = pred_judge['letter']
        elif PREFER_MODEL_B_ON_DISAGREEMENT and pred_b['option_id'] is not None:
            chosen = pred_b['option_id']
            decision = 'judge_failed_prefer_b'
            chosen_letter = pred_b['letter']
        elif pred_a['option_id'] is not None:
            chosen = pred_a['option_id']
            decision = 'judge_failed_prefer_a'
            chosen_letter = pred_a['letter']
        else:
            chosen = question.options[0].id
            decision = 'judge_failed_fallback_first_option'
            chosen_letter = 'A'
    elif PREFER_MODEL_B_ON_DISAGREEMENT and pred_b['option_id'] is not None:
        chosen = pred_b['option_id']
        decision = 'models_disagree_prefer_b'
        chosen_letter = pred_b['letter']
    elif pred_a['option_id'] is not None:
        chosen = pred_a['option_id']
        decision = 'models_disagree_prefer_a'
        chosen_letter = pred_a['letter']
    else:
        chosen = question.options[0].id
        decision = 'fallback_first_option'
        chosen_letter = 'A'

    return {
        'chosen_option_id': chosen,
        'chosen_letter': chosen_letter,
        'decision': decision,
        'pred_a': pred_a,
        'pred_b': pred_b,
        'pred_judge': pred_judge,
    }


## 9. Full Game Run

This cell starts one real text-mode game and submits model answers. The printed output is intentionally compact: level, selected option, decision rule, correctness, and timing.

In [ ]:
# Played, the real game is. Compact, the output remains.
from datetime import UTC, datetime

def play_full_game():
    run_log = []
    game = client.game.start(competition_id=COMPETITION_ID, mode=GAME_MODE)
    print(f'Started session: {game.session_id}')

    while game.in_progress and game.current_level <= MAX_LEVELS_TO_PLAY:
        question = game.current_question
        if question is None:
            break

        time_before_s = game.time_remaining
        decision = choose_answer(question)
        chosen_id = decision['chosen_option_id']
        result = game.answer(chosen_id)

        pred_b_time = decision['pred_b']['elapsed_s'] if decision['pred_b'] else 0.0
        pred_judge_time = decision.get('pred_judge', {}).get('elapsed_s', 0.0) if decision.get('pred_judge') else 0.0
        model_time_s = decision['pred_a']['elapsed_s'] + pred_b_time + pred_judge_time

        print(
            f"Level {game.current_level} | "
            f"chosen={decision['chosen_letter']}:{chosen_id} | "
            f"decision={decision['decision']} | "
            f"correct={result.correct} | "
            f"timeout={result.timed_out} | "
            f"model_time={model_time_s:.2f}s | "
            f"earned={result.earned_amount}"
        )

        run_log.append({
            'timestamp': datetime.now(UTC).isoformat(),
            'session_id': game.session_id,
            'level': game.current_level,
            'question_id': question.id,
            'question_text': question.text,
            'options': [{'id': opt.id, 'text': opt.text} for opt in question.options],
            'time_remaining_before_answer_s': time_before_s,
            'chosen_option_id': chosen_id,
            'chosen_letter': decision['chosen_letter'],
            'decision': decision,
            'correct': result.correct,
            'timed_out': result.timed_out,
            'game_over': result.game_over,
            'earned_amount': result.earned_amount,
        })

        if result.game_over or result.timed_out:
            break

        time.sleep(DELAY_BETWEEN_QUESTIONS_S)

    print(f'Final level: {game.current_level}')
    print(f'Final earned amount: {game.earned_amount}')
    return game, run_log

if RUN_FULL_GAME:
    final_game, run_log = play_full_game()
else:
    final_game, run_log = None, []
    print('Full game skipped by settings.')


## 10. Save Results

The log is saved as JSON for later error analysis and for reporting model performance in the final notebook.

In [ ]:
# Saved, the run is. Analyzed later, errors can be.
import json

if SAVE_RUN_LOG and run_log:
    log_dir = Path(PROJECT_DIR) / 'runs'
    log_dir.mkdir(parents=True, exist_ok=True)
    output_path = log_dir / f'run_{datetime.now(UTC).strftime("%Y%m%d_%H%M%S")}.json'
    with output_path.open('w', encoding='utf-8') as f:
        json.dump(run_log, f, indent=2, ensure_ascii=False)
    print('Saved run log:', output_path)
else:
    print('No run log saved.')

## 11. Leaderboard

This retrieves the leaderboard for the selected competition and mode after the run.

In [ ]:
# Checked, the leaderboard is. Compared, our run can be.
leaderboard = client.leaderboard.get(competition_id=COMPETITION_ID, limit=10, mode=GAME_MODE)

print('Leaderboard:', leaderboard.competition.name)
for rank, entry in enumerate(leaderboard.entries, 1):
    marker = ' <-- us' if entry.username == USERNAME else ''
    print(f'{rank}. {entry.username}: {entry.score} | level={entry.reached_level}{marker}')

## 12. Discussion Notes

Points to discuss in the final version:

- Whether the stronger quantized model improves over the smaller baseline.
- Whether category-specific prompts improve answer quality.
- Whether the calculator helps on Maths questions.
- Whether the two-model ensemble improves over a single model.
- How often the models disagree.
- Whether larger model B is worth the extra latency.
- Whether answers fit inside the 30-second timeout.
- Which question types produce mistakes.
- Why RAG was excluded from this baseline.
- Why speech mode was not used as the main system: the API already exposes text questions, while speech requires ASR and adds latency.